# BrowserGym Runner fuer GitLab Task 44

Dieses Notebook ist der Schritt nach dem direkten Playwright-Minimalrunner. Es nutzt BrowserGym fuer Browser-Environment, Observation und Action-Ausfuehrung, schreibt aber weiterhin die WebArena-Verified-Artefakte:

- `agent_response.json`
- `network.har`
- danach `eval_result.json` durch WebArena-Verified

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

## 1. Demo-GitLab muss laufen

Falls nicht:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

In [ ]:
subprocess.run(['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], check=True)

## 2. Task-Input erzeugen

In [ ]:
(OFFICIAL_REPO / 'output').mkdir(exist_ok=True)
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', '44',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/tasks.demo.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((OFFICIAL_REPO / 'output/tasks.demo.json').read_text())

## 3. BrowserGym-Runner starten

Der Runner benutzt `browsergym/openended`, loggt sich in GitLab ein, fuehrt eine BrowserGym-Action `goto(...)` aus und evaluiert danach.

In [ ]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_browsergym_gitlab_task44_runner.py'),
    '--repo-root', str(OFFICIAL_REPO),
    '--tasks-file', 'output/tasks.demo.json',
    '--task-id', '44',
    '--output-root', 'output/browsergym-run',
    '--config', 'examples/configs/config.demo.json',
], cwd=PROJECT_ROOT, check=True)

## 4. Ergebnis inspizieren

In [ ]:
run_dir = OFFICIAL_REPO / 'output/browsergym-run/44'
sorted(p.name for p in run_dir.iterdir())

In [ ]:
json.loads((run_dir / 'agent_response.json').read_text())

In [ ]:
eval_result = json.loads((run_dir / 'eval_result.json').read_text())
{key: eval_result.get(key) for key in ['task_id', 'status', 'score']}

## Einordnung

Wenn dieser Run `score = 1.0` erreicht, ist BrowserGym erfolgreich zwischen Task-Input und WebArena-Verified-Evaluation eingebaut. Danach ist AgentLab der naechste Runner-Layer: nicht mehr nur ein einzelner scripted Agent, sondern wiederholbare Experimente mit Result-Struktur.